# DocMax — Colab Cloud Server

Runs the real DocMax Cloud Engine (`docmax.server`, the same FastAPI app deployed on Vercel) on a Colab VM and exposes it through a Cloudflare Tunnel.

**Why Colab instead of Vercel:** the three cloud-capable tools (`compress`, `convert`, `ocr`) each need a real system binary — Ghostscript, Pandoc, Tesseract+Poppler. Vercel's Python Functions have no way to install those (confirmed by probing the sandbox directly: no apt, and `dnf`/`microdnf` exist but the filesystem is read-only and rebuilt on every cold start). Colab gives you a real, `apt-get`-able Linux VM, so this is the notebook that actually runs those tools.

**Remote MCP (M11):** the server now also speaks MCP over HTTP at `/v1/mcp` — the same `docmax.server` app, so this notebook needs no changes to serve it; the `server` extra installed in step 3 already pulls in the `mcp` SDK. An MCP client authenticates with the same bearer API key as every other endpoint (see [ADR 0035](docs/adr/0035-remote-mcp-is-a-transport-bridge-over-the-cloud-server.md)).

**Before you rely on this, know the trade-offs:**
- **Not persistent.** Colab free-tier disconnects on idle (~90 min) and has a hard session ceiling (~12h). Closing the tab kills the server and the tunnel. This is a test/demo rig, not production hosting.
- **Storage is in-memory**, same as the Vercel deployment — uploads and job state vanish on restart. That matches the project's own contract (documents are deleted on completion), it's not a shortcut taken here.
- **One tunnel hostname = one backend.** If `docmax.punith.tech` already points at the Vercel deployment (an `A` record to `76.76.21.21`), reuse a *different* hostname here (e.g. `docmax-colab.punith.tech`) — Cloudflare can't route the same name to two different origins at once.

## 0. One-time setup: Colab Secrets (recommended)

Skip the manual copy-paste on every run by storing two values once, in Colab's own Secrets manager — click the **key icon (🔑) in the left sidebar**:

| Secret name | Value |
|---|---|
| `DOCMAX_API_KEY` | Any string you choose — this becomes the server's fixed API key, so your local `docmax` config never needs updating again across restarts. |
| `CLOUDFLARE_TUNNEL_TOKEN` | The token from your Cloudflare tunnel's install command (see section 6 below if you haven't created one yet). |

For each one: **Add new secret**, paste the name and value, then toggle **Notebook access** on. Once both exist, every cell below picks them up automatically — no more pasting a token or copying a freshly-generated key out of the logs each time you restart the runtime.

Don't want to bother with Secrets? Everything below still works without them — you'll just be prompted for the tunnel token and shown a new random API key every run, same as before.

In [ ]:
# @title 1. Configuration { display-mode: "form" }
REPO_URL = "https://github.com/megabyte44/DocmaxV3.git"  # @param {type:"string"}
PROJECT_DIR = "/content/docmax-server"  # @param {type:"string"}
PORT = 8000  # @param {type:"integer"}
API_KEY = ""  # @param {type:"string"}
#: Only used if there's no DOCMAX_API_KEY Colab Secret (see section 0 above).
#: Leave blank to auto-generate a random one (printed once, in the "start server" cell).

PUBLIC_HOSTNAME = "docmax-colab.punith.tech"  # @param {type:"string"}
#: The hostname you map to this tunnel in Cloudflare Zero Trust (see the setup
#: cell below). Must NOT be a hostname already pointed at another backend.

In [ ]:
import os
import subprocess


def run(cmd, **kw):
    print(f"\n▶ {cmd}")
    subprocess.run(cmd, shell=True, check=True, **kw)


def colab_secret(name):
    """The named Colab Secret, or None if it's not set / access wasn't granted."""
    try:
        from google.colab import userdata

        return userdata.get(name) or None
    except Exception:
        return None


_banner_lines = ["DOCMAX CLOUD SERVER", "Google Colab + Cloudflare"]
_bw = max(len(l) for l in _banner_lines) + 4
print("╔" + "═" * _bw + "╗")
for _l in _banner_lines:
    print("║" + _l.center(_bw) + "║")
print("╚" + "═" * _bw + "╝")

╔═════════════════════════════╗
║     DOCMAX CLOUD SERVER     ║
║  Google Colab + Cloudflare  ║
╚═════════════════════════════╝


## 2. Clone the project

In [ ]:
print("📦 Setting up DocMax...")

if os.path.isdir(PROJECT_DIR):
    run(f"git -C {PROJECT_DIR} pull")
else:
    run(f"git clone --depth 1 {REPO_URL} {PROJECT_DIR}")

📦 Setting up DocMax...

▶ git clone --depth 1 https://github.com/megabyte44/DocmaxV3.git /content/docmax-server


## 3. Install system + Python dependencies

In [ ]:
print("🔧 Installing system binaries (Ghostscript, Tesseract, Poppler, Pandoc)...")
run("apt-get update -qq")
run("apt-get install -y -qq ghostscript tesseract-ocr poppler-utils pandoc")

🔧 Installing system binaries (Ghostscript, Tesseract, Poppler, Pandoc)...

▶ apt-get update -qq

▶ apt-get install -y -qq ghostscript tesseract-ocr poppler-utils pandoc


In [ ]:
print("📚 Installing Python dependencies (pulls opencv, pandas, etc. — a couple of minutes)...")
#: pyproject.toml's `server` extra already implies `[all]` (ocr+tables+images+
#: tui+crypto), so this one extra is everything the three cloud tools need.
run(f'pip install -q "{PROJECT_DIR}[server]"')

📚 Installing Python dependencies (pulls opencv, pandas, etc. — a couple of minutes)...

▶ pip install -q "/content/docmax-server[server]"


## 4. Install `cloudflared`

In [ ]:
import platform

print("☁️ Installing Cloudflare connector...")

arch = "amd64" if platform.machine() in ("x86_64", "AMD64") else "arm64"
run(
    f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
    f"cloudflared-linux-{arch} -O /usr/local/bin/cloudflared"
)
run("chmod +x /usr/local/bin/cloudflared")
run("cloudflared --version")

☁️ Installing Cloudflare connector...

▶ wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared

▶ chmod +x /usr/local/bin/cloudflared

▶ cloudflared --version


## 5. Start the DocMax server

In [ ]:
import secrets
import time

import requests

_secret_key = colab_secret("DOCMAX_API_KEY")
if _secret_key:
    API_KEY = _secret_key
    print(f"Using API key from the 'DOCMAX_API_KEY' Colab Secret (…{API_KEY[-4:]}).")
    print("This one stays the same across restarts — your local docmax config only needs it once.")
elif API_KEY:
    print(f"Using API key from the Configuration cell (…{API_KEY[-4:]}).")
else:
    API_KEY = secrets.token_urlsafe(32)
    print(
        "No DOCMAX_API_KEY Colab Secret and no key set above — generated one for this session only:"
    )
    print(f"  {API_KEY}")
    print(
        "It will be different next restart. Add it as a Colab Secret (see section 0) to stop seeing this."
    )

env = os.environ.copy()
#: docmax.server is deliberately excluded from the installable wheel (it ships
#: from a checkout, not PyPI — see pyproject.toml). PYTHONPATH pointed at src/
#: makes the real, complete source tree win over whatever partial `docmax`
#: package `pip install .[server]` put in site-packages, the same fix used for
#: the Vercel deployment's entrypoint.
env["PYTHONPATH"] = f"{PROJECT_DIR}/src"
env["DOCMAX_SERVER_HOST"] = "0.0.0.0"
env["DOCMAX_SERVER_PORT"] = str(PORT)
env["DOCMAX_SERVER_API_KEYS"] = API_KEY

log_path = "/content/docmax_server.log"
log_file = open(log_path, "w")
server = subprocess.Popen(
    ["python", "-m", "docmax.server"],
    cwd=PROJECT_DIR,
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("\nWaiting for the server to come up...")
for attempt in range(30):
    try:
        r = requests.get(f"http://127.0.0.1:{PORT}/healthz", timeout=2)
        if r.status_code == 200:
            print(f"✅ DocMax server is up: {r.json()}")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)
else:
    print("❌ Server did not come up in time. Log tail:")
    log_file.flush()
    run(f"tail -n 60 {log_path}")
    raise RuntimeError("DocMax server failed to start — see the log tail above")

Using API key from the 'DOCMAX_API_KEY' Colab Secret (…sj7w).
This one stays the same across restarts — your local docmax config only needs it once.

Waiting for the server to come up...
✅ DocMax server is up: {'ok': True, 'api_version': '1'}


## 6. One-time Cloudflare Tunnel setup

Skip this if you already have a tunnel configured.

1. Open the [Cloudflare Zero Trust dashboard](https://one.dash.cloudflare.com/) → **Networks → Tunnels → Create a tunnel**.
2. Choose **Cloudflared**, name it (e.g. `docmax-colab`), and copy the token from the install command it shows you — the long string after `--token`.
3. Under **Public Hostname**, add one:
   - Subdomain: whatever you set `PUBLIC_HOSTNAME` to above (e.g. `docmax-colab`)
   - Domain: your domain (e.g. `punith.tech`)
   - Service: `HTTP` → `localhost:8000` (or whatever `PORT` is set to)
4. Either save this token as the `CLOUDFLARE_TUNNEL_TOKEN` Colab Secret (section 0 — recommended, no more re-pasting), or just paste it into the next cell when it prompts. A pasted token is only held in memory for this session and is never written to disk.

In [ ]:
import getpass

TUNNEL_TOKEN = colab_secret("CLOUDFLARE_TUNNEL_TOKEN")
if TUNNEL_TOKEN:
    print("Using tunnel token from the 'CLOUDFLARE_TUNNEL_TOKEN' Colab Secret.")
else:
    TUNNEL_TOKEN = getpass.getpass("Cloudflare tunnel token: ")

tunnel_log = open("/content/cloudflared.log", "w")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "run", "--token", TUNNEL_TOKEN],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    text=True,
)
del TUNNEL_TOKEN  # out of memory as soon as it's handed to the process

print("Waiting for the tunnel to register...")
time.sleep(8)

public_url = f"https://{PUBLIC_HOSTNAME}/healthz"
try:
    r = requests.get(public_url, timeout=20)
    print(f"✅ Public endpoint reachable: {r.status_code} {r.json()}")
except Exception as e:
    print(f"⚠️ Not reachable yet ({e}).")
    print("DNS/tunnel propagation can take a minute — rerun this cell to retry.")

Using tunnel token from the 'CLOUDFLARE_TUNNEL_TOKEN' Colab Secret.
Waiting for the tunnel to register...
✅ Public endpoint reachable: 200 {'ok': True, 'api_version': '1'}


## 7. Ready

In [ ]:
#: Built from the actual values rather than hand-drawn ASCII art, so the box
#: can't end up misaligned or leak an unsubstituted {placeholder}.
summary_lines = [
    "DOCMAX IS READY",
    "",
    f"Local:   http://127.0.0.1:{PORT}",
    f"Public:  https://{PUBLIC_HOSTNAME}",
    f"API key: …{API_KEY[-4:]}   (full value was printed once, above)",
    "",
    f"curl https://{PUBLIC_HOSTNAME}/v1/capabilities \\",
    '  -H "Authorization: Bearer <your key>"',
    "",
    f"Remote MCP (M11): https://{PUBLIC_HOSTNAME}/v1/mcp",
    "  same bearer key, streamable-HTTP transport -- see ADR 0035",
    "",
    "Keep this Colab tab open and the runtime connected —",
    "closing it kills both the server and the tunnel.",
]
_width = max(len(l) for l in summary_lines) + 2
print("╔" + "═" * _width + "╗")
for _l in summary_lines:
    print("║ " + _l.ljust(_width - 1) + "║")
print("╚" + "═" * _width + "╝")

In [ ]:
!tail -f -n 40 /content/docmax_server.log /content/cloudflared.log

==> /content/docmax_server.log <==
INFO:     Started server process [3797]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:44526 - "GET /healthz HTTP/1.1" 200 OK
INFO:     35.184.121.127:0 - "GET /healthz HTTP/1.1" 200 OK
INFO:     152.57.224.132:0 - "POST /v1/tools/compress HTTP/1.1" 200 OK
INFO:     152.57.224.132:0 - "GET /v1/outputs/f_dd753581f1352774422c1576 HTTP/1.1" 200 OK
INFO:     152.57.224.132:0 - "POST /v1/tools/ocr HTTP/1.1" 200 OK
INFO:     152.57.224.132:0 - "POST /v1/tools/ocr HTTP/1.1" 200 OK
INFO:     152.57.224.132:0 - "GET /v1/outputs/f_89547e3c6ac7ab9571170e51 HTTP/1.1" 200 OK

==> /content/cloudflared.log <==
2026-08-30T15:24:23Z INF Initial protocol quic
2026-08-30T15:24:23Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
2026-08-30T15:24:23Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026/08/30 15:24:23 failed to su